# MS MARCO Multi-Stage Ranker — Colab Pipeline

This notebook runs the complete end-to-end information retrieval portfolio pipeline:
1. **Data Ingestion & Cleaning**: MS MARCO v1.1 subset streaming, language filtering, exact & MinHash near-duplicate deduplication.
2. **Lexical Retrieval**: BM25 Okapi baseline with NDCG@10, MRR@10, Recall@100 evaluation.
3. **Dense & Hybrid Retrieval**: MiniLM embeddings + FAISS Flat/IVF indexing + Reciprocal Rank Fusion (RRF).
4. **Cross-Encoder Reranking**: Zero-shot vs Random Negatives vs BM25 Hard Negatives across rerank depths (20, 50, 100).
5. **Error Analysis & Report Generation**: Qualitative error inspection on the hardest queries and exportable markdown reports.

> **Prerequisite**: Set **Runtime → Change runtime type → T4 GPU** before running.

### Step 1: Verify Hardware Accelerator

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

### Step 2: Upload Project Archive & Install Dependencies

In [ ]:
from google.colab import files
import os

# Upload msmarco-ranker-colab.zip from your local workspace
uploaded = files.upload()
assert 'msmarco-ranker-colab.zip' in uploaded, "Please upload 'msmarco-ranker-colab.zip'"

!rm -rf /content/msmarco-ranker
!unzip -q msmarco-ranker-colab.zip -d /content/msmarco-ranker
%cd /content/msmarco-ranker
!pip -q install -r requirements.txt

### Step 3: Run Unit Test Suite
Validates NDCG@10, MRR@10, Recall@100 metric calculations and RRF fusion logic.

In [ ]:
%cd /content/msmarco-ranker
!pytest -v

### Step 4: Data Preparation
Streams MS MARCO v1.1 from Hugging Face, cleans empty/short and non-English passages, deduplicates with exact normalized hashes and MinHash LSH, and saves Parquet splits.

> **Tip**: Use `configs/debug.yaml` for a fast 2-minute smoke test, or `configs/default.yaml` for the full benchmark.

In [ ]:
%cd /content/msmarco-ranker

# Choose CONFIG = 'configs/debug.yaml' for quick test or 'configs/default.yaml' for full benchmark
CONFIG = 'configs/default.yaml'

!python scripts/prepare_data.py --config {CONFIG}

### Step 5: Lexical Retrieval (BM25 Okapi Baseline)

In [ ]:
%cd /content/msmarco-ranker
!python scripts/run_bm25.py --config {CONFIG}

### Step 6: Dense Retrieval & RRF Hybrid Fusion
Encodes the corpus with `all-MiniLM-L6-v2`, builds FlatIP & IVF-Flat FAISS indices, and executes Reciprocal Rank Fusion.

In [ ]:
%cd /content/msmarco-ranker
!python scripts/run_dense.py --config {CONFIG}

### Step 7: Cross-Encoder Fine-Tuning & Reranking Ablations
Evaluates:
- Zero-shot / pretrained cross-encoder baseline
- Fine-tuning with random negatives
- Fine-tuning with BM25 hard negatives
- Reranking depths: 20, 50, 100

In [ ]:
%cd /content/msmarco-ranker
!python scripts/run_reranker.py --config {CONFIG}

### Step 8: Generate Reports & Display Results

In [ ]:
%cd /content/msmarco-ranker
!python scripts/generate_report.py

from IPython.display import Markdown, display
with open('reports/data_quality.md') as f:
    display(Markdown(f.read()))

with open('reports/results.md') as f:
    display(Markdown(f.read()))

### Step 9: Qualitative Error Analysis
Inspect the 20 lowest-scoring queries from the best reranker run to understand failure modes.

In [ ]:
%cd /content/msmarco-ranker
import glob
rankings_files = glob.glob('artifacts/rankings_rerank_hard_negatives_depth_100.csv') or glob.glob('artifacts/rankings_*.csv')
if rankings_files:
    target = rankings_files[0]
    print(f"Analyzing worst queries for: {target}")
    !python scripts/worst_queries.py --rankings {target}
else:
    print("No rankings file found yet.")

### Step 10: Download Artifacts & Reports to Local Machine

In [ ]:
%cd /content/msmarco-ranker
from google.colab import files

# Package reports, configs, and CSV rankings
!zip -r /content/msmarco-ranker-results.zip reports/ artifacts/*.csv configs/
print("Downloading results archive...")
files.download('/content/msmarco-ranker-results.zip')